In [1]:
from huggingface_hub import list_repo_files

files = list_repo_files("IlyaGusev/ru_turbo_alpaca", repo_type="dataset")
for f in files:
    print(f)

.gitattributes
README.md
ru_turbo_alpaca.jsonl.zst
ru_turbo_alpaca.py


In [2]:
from datasets import load_dataset

ds = load_dataset("IlyaGusev/ru_turbo_alpaca", revision="refs/convert/parquet", split="train")
print(ds)
print()
for i in [0, 1]:
    for key, value in ds[i].items():
        print(f"{key}: {value!r}")
    print("-" * 40)

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 27.5MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'alternative_output', 'label', 'all_labels', 'agreement', 'overlap'],
    num_rows: 29822
})

instruction: 'Опишите, как сделать горшок из глины.'
input: ''
output: 'Для изготовления горшка из глины сначала необходимо взять глину и размешать ее водой до состояния пластилина. Затем нужно сформировать глиняный комок нужной формы и тщательно откачать влагу из глины, чтобы она стала твердой. Горшок нужно обжечь в духовке при высокой температуре, чтобы он стал прочным и непроницаемым.'
alternative_output: 'Для изготовления горшка из глины нужно выполнить следующие шаги:\n\n1. Подготовить глину: глину необходимо хорошо промыть и измельчить. Если глина слишком сухая, ее нужно замочить.\n\n2. Сформировать горшок: возьмите кусок глины и разминайте его в руках, сделав шарик. Потом начинайте вращать шарик между ладонями, чтоб сформировался цилиндр. Затем используйте пальцы, чтобы создать стенки горшка высотой и диаметром по желанию.\n\n3.

In [3]:
from collections import Counter

print(Counter(ds["label"]))

Counter({None: 26016, 'ok': 2905, 'bad_output': 690, 'bad_task': 211})


In [4]:
import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from types import SimpleNamespace
from contextlib import nullcontext
from tokenizers import Tokenizer

TOK_DIR, CKPT_PATH = None, None
for root, dirs, files in os.walk("/kaggle/input"):
    if "tokenizer.json" in files:
        TOK_DIR = root
    if "ckpt.pt" in files:
        CKPT_PATH = os.path.join(root, "ckpt.pt")
print("tokenizer:", TOK_DIR)
print("checkpoint:", CKPT_PATH)

tok = Tokenizer.from_file(f"{TOK_DIR}/tokenizer.json")
eot_id = tok.token_to_id("<|endoftext|>")
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer: /kaggle/input/notebooks/andruy28/good-nlp-1b
checkpoint: /kaggle/input/notebooks/andruy28/good-nlp-2b/ckpt.pt


In [5]:
assert "instruction" in ds.column_names and "output" in ds.column_names, ds.column_names
BLOCK = 256

def build_prompt(instruction, inp=""):
    text = "Пользователь: " + instruction.strip()
    if inp and inp.strip():
        text += "\n" + inp.strip()
    return text + "\nАссистент:"

X_list, Y_list = [], []
n_skipped = 0
n_bad = 0
for ex in ds:
    if ex["label"] == "bad_output":
        n_bad += 1
        continue
    prompt_ids = tok.encode(build_prompt(ex["instruction"], ex.get("input", ""))).ids
    answer_ids = tok.encode(" " + ex["output"].strip()).ids + [eot_id]
    ids = prompt_ids + answer_ids
    if len(ids) > BLOCK + 1:
        n_skipped += 1
        continue
    x = ids[:-1]
    y = ids[1:]
    n_mask = len(prompt_ids) - 1
    y[:n_mask] = [-100] * n_mask
    pad = BLOCK - len(x)
    X_list.append(x + [eot_id] * pad)
    Y_list.append(y + [-100] * pad)

X = torch.tensor(X_list, dtype=torch.long)
Y = torch.tensor(Y_list, dtype=torch.long)

g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(X), generator=g)
n_val = len(X) // 20
X_val, Y_val = X[perm[:n_val]], Y[perm[:n_val]]
X_train, Y_train = X[perm[n_val:]], Y[perm[n_val:]]

print(f"Всего примеров: {len(ds)}, отброшено длинных: {n_skipped}")
print(f"train: {len(X_train)}, val: {len(X_val)}")
print(f"Доля токенов, по которым считается loss: {(Y != -100).float().mean():.1%}")
print()
print("Пример x:", tok.decode([t for t in X_train[0].tolist()]))
print("Что учим (y без маски):", tok.decode([t for t in Y_train[0].tolist() if t != -100]))

Всего примеров: 29822, отброшено длинных: 436
train: 27262, val: 1434
Доля токенов, по которым считается loss: 28.7%

Пример x: Пользователь: Напиши новостную статью о событиях в городе за прошлую неделю.
Ассистент: За прошедшую неделю в городе произошло несколько событий. В понедельник прошел митинг против увольнений на одном из крупных предприятий. Во вторник произошло ЧП на крыше высотного здания, когда рабочий провалился сквозь лед и пришел в тяжелый ожог. В среду открылась новая библиотека в центре города. В четверг был запущен новый транспортный проект - метрополитен. В пятницу в городском парке состоялся концерт известной рок-группы.
Что учим (y без маски):  За прошедшую неделю в городе произошло несколько событий. В понедельник прошел митинг против увольнений на одном из крупных предприятий. Во вторник произошло ЧП на крыше высотного здания, когда рабочий провалился сквозь лед и пришел в тяжелый ожог. В среду открылась новая библиотека в центре города. В четверг был запущен нов

In [6]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.dropout = cfg.dropout
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


ckpt = torch.load(CKPT_PATH, map_location=device)
cfg = SimpleNamespace(**ckpt["config"])
cfg.dropout = 0.1
model = GPT(cfg).to(device)
model.load_state_dict(ckpt["model"])
n_params = sum(p.numel() for p in model.parameters())
print(f"Загружена модель: {n_params / 1e6:.1f}M параметров, val loss предобучения {ckpt['val_loss']:.3f}")

Загружена модель: 33.5M параметров, val loss предобучения 2.951


In [7]:
@torch.no_grad()
def chat(question, max_new_tokens=100, temperature=0.7, top_k=50):
    model.eval()
    idx = torch.tensor([tok.encode(build_prompt(question)).ids], dtype=torch.long, device=device)
    start = idx.shape[1]
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -cfg.block_size:])
        logits = logits[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")
        next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
        if next_id.item() == eot_id:
            break
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return tok.decode(idx[0, start:].tolist())

test_questions = [
    "Назови три крупные реки России.",
    "Как заварить чай?",
    "Что такое фотосинтез?",
]
torch.manual_seed(0)
for q in test_questions:
    print("Вопрос:", q)
    print("Ответ:", chat(q))
    print()

Вопрос: Назови три крупные реки России.
Ответ:  А. П. Бородавкин
Ассистент: Н. В. Гоголь
Ассистент: Н. В. Гоголь
Ассистент: О. Г. Кузьмин
Ассистент: Н. И. Четверикова
Ассистент: Н. Г. Кузьмин
Ассистент: Н. В.

Вопрос: Как заварить чай?
Ответ:  Как заварить чай?
Проклятый: Как заварить чай?
Продавец: Как заварить чай!

Приставки

Дополнения
Версия 1.5.0.0.1.1.0.1.1.1.1.1.1.0.1.0.1.1.1.1.1.1.1.1.1.1.1.1

Вопрос: Что такое фотосинтез?
Ответ:  Вершина: Дело
Ассистент: Дело
Ассистент: Бумажный
Ассистент: Неверие: Пифия
Ассистент: Идут ли вы какие-либо снимки?
Ассистент: Разочарованный
Ассистент: Ты не пой?
Ассистент: Подставка, но какая-нибудь картина?
Ассистент: Я не видел



In [8]:
batch_size = 32
epochs = 4
max_iters = epochs * len(X_train) // batch_size
eval_interval = 200
warmup_iters = 100
max_lr, min_lr = 1e-4, 1e-5
use_amp = device == "cuda"

def get_lr(it):
    if it < warmup_iters:
        return max_lr * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay_params, "weight_decay": 0.1},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=max_lr, betas=(0.9, 0.95),
)
ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp else nullcontext()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

@torch.no_grad()
def val_loss():
    model.eval()
    losses = []
    for i in range(0, len(X_val), batch_size):
        xb, yb = X_val[i:i + batch_size].to(device), Y_val[i:i + batch_size].to(device)
        with ctx:
            _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

print(f"Шагов: {max_iters}")
model.train()
t0 = time.time()
for it in range(max_iters + 1):
    lr = get_lr(it)
    for group in optimizer.param_groups:
        group["lr"] = lr

    if it % eval_interval == 0 or it == max_iters:
        print(f"шаг {it}: val loss {val_loss():.3f}, lr {lr:.2e}, прошло {time.time() - t0:.0f} с", flush=True)
        print("  >", chat(test_questions[0]), flush=True)
    if it == max_iters:
        break

    ix = torch.randint(len(X_train), (batch_size,))
    xb, yb = X_train[ix].to(device), Y_train[ix].to(device)
    with ctx:
        _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

torch.save({"model": model.state_dict(), "config": vars(cfg)}, "/kaggle/working/ckpt_sft.pt")
print("Сохранено.")

Шагов: 3407
шаг 0: val loss 3.132, lr 1.00e-06, прошло 2 с
  >  Улица Маршала Жукова.
Мастер: Улица Маршала Жукова.
Ассистент: Улица Маршала Жукова.

Публикации
Миссия. События. — М.: Мир, 1991. — 352 с. — ISBN 5-7223-0029-1
Миссия. — М.: Мир, 1991. — 352 с. — ISBN 5-7224-
шаг 200: val loss 2.656, lr 9.98e-05, прошло 33 с
  >  Азов - это река, река в России и России, впадает в Азовскую бухту, река протекает в Алтайский край, река относится к бассейну Каспия и имеет ширину до 18 км. В Азовское море Азовское море и Азовское море очень богатыми водами. Азовское море является одним из самых загрязнённых городов России и входит в список самых загрязненных городов мира.
шаг 400: val loss 2.537, lr 9.82e-05, прошло 65 с
  >  Бассейн реки России
шаг 600: val loss 2.469, lr 9.50e-05, прошло 102 с
  >  Река России представляет собой реку, окружающую реку. Вода богата большим количеством океанов, таких как лесные луга и т.д. Также есть множество водопадов и овраждений, которые окружают реки. Неко

In [9]:
ckpt = torch.load("/kaggle/working/ckpt_sft.pt", map_location=device)
cfg = SimpleNamespace(**ckpt["config"])
model = GPT(cfg).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

GPT(
  (tok_emb): Embedding(16000, 512)
  (pos_emb): Embedding(256, 512)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-7): 8 x Block(
      (ln1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=512, out_features=1536, bias=False)
        (proj): Linear(in_features=512, out_features=512, bias=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=False)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=False)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=512, out_features=16000, bias=False)
)

In [10]:
torch.manual_seed(0)
for q in test_questions:
    print("Вопрос:", q)
    print("Ответ:", chat(q))
    print()

Вопрос: Назови три крупные реки России.
Ответ:  1. У реки России есть одна река, самая большая из которых расположена в России.
2. Природные ресурсы, такие как реки, озера, озера, озера, озера, озерки и многие другие, являются важными важными для развития нашей планеты.
3. В России есть множество красивых парков и деревень, включая Красную площадь, Собор Василия Блаженного, Собор Василия Блаженного и другие.
4. В России есть несколько знаменитых парков,

Вопрос: Как заварить чай?
Ответ:  Для заваривания чай необходимо сварить чай на среднем огне, затем добавить чай и добавить сахар, сахар и сахар в чай. Затем добавить сахар и сахар с помощью кофейных капель и добавить их в чай.

Вопрос: Что такое фотосинтез?
Ответ:  К фотосинтез - это процесс, при котором изученные растения используют энергию света для создания растений, которые содержат белки, цветы и растения. Когда фотосинтез проходит через растения, фотосинтез происходит в условиях высокой влажности и влажности.

